In [1]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
data = 'weatherAUS.csv'
df = pd.read_csv(data)

In [4]:
df.shape

(145460, 23)

In [5]:
df.head(10)

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,12/1/2008,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,12/2/2008,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,12/3/2008,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,12/4/2008,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,12/5/2008,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No
5,12/6/2008,Albury,14.6,29.7,0.2,NaN,NaN,WNW,56.0,W,...,55.0,23.0,1009.2,1005.4,NaN,NaN,20.6,28.9,No,No
6,12/7/2008,Albury,14.3,25.0,0.0,NaN,NaN,W,50.0,SW,...,49.0,19.0,1009.6,1008.2,1.0,NaN,18.1,24.6,No,No
7,12/8/2008,Albury,7.7,26.7,0.0,NaN,NaN,W,35.0,SSE,...,48.0,19.0,1013.4,1010.1,NaN,NaN,16.3,25.5,No,No
8,12/9/2008,Albury,9.7,31.9,0.0,NaN,NaN,NNW,80.0,SE,...,42.0,9.0,1008.9,1003.6,NaN,NaN,18.3,30.2,No,Yes
9,12/10/2008,Albury,13.1,30.1,1.4,NaN,NaN,W,28.0,S,...,58.0,27.0,1007.0,1005.7,NaN,NaN,20.1,28.2,Yes,No


In [6]:
df['Date'] = pd.to_datetime(df['Date'])
df['Year'] = df['Date'].dt.year

In [7]:
df.drop('Date', axis = 1, inplace = True)

In [8]:
df.dtypes

Location          object
MinTemp          float64
MaxTemp          float64
Rainfall         float64
Evaporation      float64
Sunshine         float64
WindGustDir       object
WindGustSpeed    float64
WindDir9am        object
WindDir3pm        object
WindSpeed9am     float64
WindSpeed3pm     float64
Humidity9am      float64
Humidity3pm      float64
Pressure9am      float64
Pressure3pm      float64
Cloud9am         float64
Cloud3pm         float64
Temp9am          float64
Temp3pm          float64
RainToday         object
RainTomorrow      object
Year               int32
dtype: object

In [9]:
# cheek for outliers
df.describe().loc[['min', 'mean', 'max']]

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,Year
min,-8.500000,-4.800000,0.000000,0.000000,0.000000,6.00000,0.000000,0.000000,0.000000,0.000000,980.50000,977.100000,0.000000,0.00000,-7.200000,-5.40000,2007.000000
mean,12.194034,23.221348,2.360918,5.468232,7.611178,40.03523,14.043426,18.662657,68.880831,51.539116,1017.64994,1015.255889,4.447461,4.50993,16.990631,21.68339,2012.769751
max,33.900000,48.100000,371.000000,145.000000,14.500000,135.00000,130.000000,87.000000,100.000000,100.000000,1041.00000,1039.600000,9.000000,9.00000,40.200000,46.70000,2017.000000


In [10]:
numeric = df.select_dtypes(include='number')  # select numeric columns

outlier_indices = set()  # store all outlier row indices

for col in numeric:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    # find outliers for this column
    outliers = df[(df[col] < (q1 - 1.5 * iqr)) | (df[col] > (q3 + 1.5 * iqr))].index
    outlier_indices.update(outliers)

# drop all outliers at once (in place)
df.drop(outlier_indices, inplace=True)

In [11]:
df.shape

(112175, 23)

In [12]:
df.isnull().sum()

Location             0
MinTemp           1307
MaxTemp           1099
Rainfall          3024
Evaporation      49007
Sunshine         54389
WindGustDir       7953
WindGustSpeed     7914
WindDir9am        8980
WindDir3pm        3349
WindSpeed9am      1494
WindSpeed3pm      2443
Humidity9am       2096
Humidity3pm       3529
Pressure9am      11889
Pressure3pm      11851
Cloud9am         44619
Cloud3pm         47299
Temp9am           1435
Temp3pm           2851
RainToday         3024
RainTomorrow      2712
Year                 0
dtype: int64

In [13]:
num_col = df.select_dtypes('number')
obj_col = df.select_dtypes('object')

for col in num_col:
    df[col].fillna(df[col].mean(), inplace = True)

for col in obj_col:
    df[col].fillna(df[col].mode()[0], inplace = True)

In [14]:
df.isnull().sum()

Location         0
MinTemp          0
MaxTemp          0
Rainfall         0
Evaporation      0
Sunshine         0
WindGustDir      0
WindGustSpeed    0
WindDir9am       0
WindDir3pm       0
WindSpeed9am     0
WindSpeed3pm     0
Humidity9am      0
Humidity3pm      0
Pressure9am      0
Pressure3pm      0
Cloud9am         0
Cloud3pm         0
Temp9am          0
Temp3pm          0
RainToday        0
RainTomorrow     0
Year             0
dtype: int64

In [15]:
x = df.drop(['RainTomorrow'], axis=1)

y = df['RainTomorrow']

In [16]:
x = pd.get_dummies(x, columns=x.select_dtypes(include=["object"]).columns, drop_first=True)

In [17]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

In [18]:
x = pd.DataFrame(x_scaled, columns=x.columns, index=x.index)

In [19]:
x.describe()

,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustSpeed,WindSpeed9am,WindSpeed3pm,Humidity9am,Humidity3pm,...,WindDir3pm_NW,WindDir3pm_S,WindDir3pm_SE,WindDir3pm_SSE,WindDir3pm_SSW,WindDir3pm_SW,WindDir3pm_W,WindDir3pm_WNW,WindDir3pm_WSW,RainToday_Yes
count,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,...,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,1.121750e+05,112175.000000,1.121750e+05,1.121750e+05,1.121750e+05,112175.000000
mean,4.053910e-16,1.945877e-16,-8.918602e-17,-7.378116e-16,1.398599e-16,-1.692507e-16,1.256712e-16,-1.743181e-16,-8.634828e-16,3.689058e-16,...,-9.121297e-18,2.533694e-17,-2.635041e-17,-1.013477e-17,-8.513211e-17,0.000000,-9.121297e-17,-8.513211e-17,-2.432346e-17,0.000000
std,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,...,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00,1.000004,1.000004e+00,1.000004e+00,1.000004e+00,1.000004
min,-2.860305e+00,-3.133884e+00,-4.022168e-01,-2.280027e+00,-3.119082e+00,-2.904117e+00,-1.640205e+00,-2.274487e+00,-2.803536e+00,-2.528310e+00,...,-2.453673e-01,-2.646995e-01,-3.445876e-01,-2.608026e-01,-2.333553e-01,-0.255241,-2.588869e-01,-2.431748e-01,-2.527155e-01,-0.235092
25%,-7.165142e-01,-7.517796e-01,-4.022168e-01,-3.752652e-01,0.000000e+00,-7.032051e-01,-7.549374e-01,-6.076960e-01,-6.382461e-01,-7.325868e-01,...,-2.453673e-01,-2.646995e-01,-3.445876e-01,-2.608026e-01,-2.333553e-01,-0.255241,-2.588869e-01,-2.431748e-01,-2.527155e-01,-0.235092
50%,-7.171540e-03,-5.203642e-02,-4.022168e-01,-7.689854e-16,0.000000e+00,-6.127248e-02,3.863212e-03,-9.483719e-02,-8.097530e-16,3.752759e-16,...,-2.453673e-01,-2.646995e-01,-3.445876e-01,-2.608026e-01,-2.333553e-01,-0.255241,-2.588869e-01,-2.431748e-01,-2.527155e-01,-0.235092
75%,7.179343e-01,7.072593e-01,-4.022168e-01,5.763506e-02,4.233679e-01,5.806602e-01,7.626638e-01,5.462363e-01,6.723239e-01,6.406132e-01,...,-2.453673e-01,-2.646995e-01,-3.445876e-01,-2.608026e-01,-2.333553e-01,-0.255241,-2.588869e-01,-2.431748e-01,-2.527155e-01,-0.235092
max,2.972068e+00,2.970258e+00,4.760793e+00,4.040318e+00,2.464127e+00,3.148391e+00,3.039066e+00,2.725886e+00,1.868931e+00,2.700413e+00,...,4.075523e+00,3.777869e+00,2.902020e+00,3.834317e+00,4.285311e+00,3.917864,3.862691e+00,4.112268e+00,3.957020e+00,4.253659


In [20]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,train_size=0.8, random_state=122)

In [23]:
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(random_state=42)
model.fit(x_train, y_train)
y_hat = model.predict(x_test)

In [24]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_hat))

              precision    recall  f1-score   support

          No       0.89      0.97      0.93     18876
         Yes       0.70      0.34      0.46      3559

    accuracy                           0.87     22435
   macro avg       0.79      0.66      0.69     22435
weighted avg       0.86      0.87      0.85     22435

